In [ ]:
with base as (

    select distinct
        ca.client_id,
        ca.campaign_name,

        case
            when ca.com_cus_sgr_desc like '%200%' then 200
            when ca.com_cus_sgr_desc like '%300%' then 300
            when ca.com_cus_sgr_desc like '%400%' then 400
            when ca.com_cus_sgr_desc like '%500%' then 500
        end as nominal

    from cvm_sbx.{prefix}_CVMB_24118_campaigns_audience ca

    join cvm_sbx.{prefix}_CVMB_24118_client_cohorts cc
        on ca.client_id = cc.client_id

    where cc.campaigns_cnt >= 2

),

client_nominal as (

    select
        client_id,
        nominal,
        count(distinct campaign_name) as nominal_cnt

    from base

    where nominal is not null

    group by
        client_id,
        nominal

)

select
    nominal,
    nominal_cnt,
    count(*) as client_cnt

from client_nominal

where nominal_cnt >= 2

group by
    nominal,
    nominal_cnt

order by
    nominal,
    nominal_cnt;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# nominal_repeat - результат SQL:
# nominal, nominal_cnt, client_cnt

plot_df = nominal_repeat.copy()

pivot_df = (
    plot_df
    .pivot_table(
        index='nominal',
        columns='nominal_cnt',
        values='client_cnt',
        aggfunc='sum',
        fill_value=0
    )
    .sort_index()
)

ax = pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6)
)

ax.set_title('Повторяемость попадания клиентов в номиналы')
ax.set_xlabel('Номинал')
ax.set_ylabel('Количество клиентов')

ax.legend(
    title='Сколько раз клиент попадал в номинал',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', ' '))
)

ax.grid(True, axis='y', alpha=0.3)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()